# Datos faltantes — educación de los padres (R2-4)

Notebook independiente para responder al Revisor 2 (comentario 4): compara el tratamiento por mediana (usado en el manuscrito) contra (A) indicadores de valor faltante y (B) imputación múltiple (MICE), y su efecto sobre las asignaciones de clúster (ARI vs. baseline). Reutiliza exactamente el mismo preprocesamiento de `pipeline_clustering_optimizado_9.ipynb`.

**Cómo correrlo:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno` → CPU (no necesita GPU).
2. Ajusta la ruta de tu `df_maestra.csv` en la celda de carga de datos si no está en `/content/drive/MyDrive/Proyecto/`.
3. Corre todas las celdas en orden (`Entorno de ejecución` → `Ejecutar todas`).
4. Los resultados se guardan automáticamente en tu Drive, en `/content/drive/MyDrive/Proyecto/datos_faltantes/`, por si la sesión se desconecta.

In [ ]:
!pip install -q umap-learn

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
import umap.umap_ as umap_cpu
import warnings
warnings.filterwarnings("ignore")
print("✅ Librerías listas")

## 1. Cargar los datos desde Google Drive

Ajusta la ruta si tu `df_maestra.csv` está en otra carpeta de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
OUT_DIR = '/content/drive/MyDrive/Proyecto/datos_faltantes'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'Técnica o tecnológica incompleta': 5,
        'Técnica o tecnológica completa': 6,
        'Educación profesional incompleta': 7,
        'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o más'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBAÑO':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']
    col_geo = 'ESTU_COD_DEPTO_PRESENTACION'

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    cols_extra = [col_geo, 'INST_COD_INSTITUCION', 'ESTU_PRGM_ACADEMICO',
                  'PERIODO', 'PUNT_GLOBAL', 'ESTU_CONSECUTIVO']
    cols_df = cols_usar + [c for c in cols_extra if c in df_limpio.columns]
    df_filtrado = df_limpio[[c for c in cols_df if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"Filas después de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"Preprocesamiento completo — shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler


In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=None)
n_total = X_full.shape[0]
print(f"Shape X_full: {X_full.shape}")

## 3. Reconstruir la máscara de valores faltantes (antes de imputar)

`FAMI_EDUCACIONPADRE` / `FAMI_EDUCACIONMADRE` tienen ~22-23% de valores faltantes cada una; el manuscrito los imputa con la mediana. Aquí reconstruimos cuáles filas fueron imputadas, para poder comparar tratamientos alternativos.

In [ ]:
MAPA_EDUC = {
    'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
    'Secundaria (Bachillerato) incompleta': 3,
    'Secundaria (Bachillerato) completa': 4,
    'Técnica o tecnológica incompleta': 5,
    'Técnica o tecnológica completa': 6,
    'Educación profesional incompleta': 7,
    'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9,
}
COLUMNAS_PUNTAJE = ['MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
                    'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
COLUMNAS_ORDINALES = ['FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
                      'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']

df_mask = df.copy()
df_mask['FAMI_EDUCACIONPADRE_num'] = df_mask['FAMI_EDUCACIONPADRE'].map(MAPA_EDUC)
df_mask['FAMI_EDUCACIONMADRE_num'] = df_mask['FAMI_EDUCACIONMADRE'].map(MAPA_EDUC)
falta_padre_all = df_mask['FAMI_EDUCACIONPADRE_num'].isna().to_numpy()
falta_madre_all = df_mask['FAMI_EDUCACIONMADRE_num'].isna().to_numpy()
puntajes = df_mask[COLUMNAS_PUNTAJE].apply(pd.to_numeric, errors='coerce')
filas_validas = puntajes.notna().all(axis=1).to_numpy()

falta_padre = falta_padre_all[filas_validas]
falta_madre = falta_madre_all[filas_validas]
print(f"% faltante padre={falta_padre.mean()*100:.1f}%  madre={falta_madre.mean()*100:.1f}%")
assert len(falta_padre) == n_total

## 4. Variante A (indicadores de faltante) y Variante B (imputación múltiple / MICE)

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import adjusted_rand_score

# Variante A: 32 columnas originales + 2 indicadores binarios
X_indicador = np.hstack([X_full, falta_padre.reshape(-1, 1).astype(float),
                          falta_madre.reshape(-1, 1).astype(float)])
print(f"X_indicador shape: {X_indicador.shape}")

def imputar_mice(seed):
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                    'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                   'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    d = df_mask.loc[filas_validas].reset_index(drop=True).copy()
    d['FAMI_ESTRATOVIVIENDA'] = d['FAMI_ESTRATOVIVIENDA'].map(mapa_estrato)
    d['ESTU_VALORMATRICULAUNIVERSIDAD'] = d['ESTU_VALORMATRICULAUNIVERSIDAD'].map(mapa_valormatricula)
    d['ESTU_HORASSEMANATRABAJA'] = d['ESTU_HORASSEMANATRABAJA'].map(mapeo_horas)
    for col in ['FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD', 'ESTU_HORASSEMANATRABAJA']:
        d[col] = d[col].fillna(d[col].median())
    punt = d[COLUMNAS_PUNTAJE].apply(pd.to_numeric, errors='coerce')
    for col in COLUMNAS_PUNTAJE:
        punt[col] = punt[col].fillna(punt[col].mean())
    predictores = pd.concat([
        d[['FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD', 'ESTU_HORASSEMANATRABAJA']],
        punt,
        d[['FAMI_EDUCACIONPADRE_num', 'FAMI_EDUCACIONMADRE_num']],
    ], axis=1)
    imputer = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True,
                                max_iter=8, random_state=seed, min_value=0, max_value=9)
    completo = imputer.fit_transform(predictores)
    padre_imp = np.clip(np.round(completo[:, -2]), 0, 9)
    madre_imp = np.clip(np.round(completo[:, -1]), 0, 9)
    return np.column_stack([
        d['FAMI_ESTRATOVIVIENDA'].values, d['ESTU_VALORMATRICULAUNIVERSIDAD'].values,
        padre_imp, madre_imp, d['ESTU_HORASSEMANATRABAJA'].values,
    ]).astype(float)

def correr_pipeline(X, seed, idx_fit, idx_eval):
    reducer = umap_cpu.UMAP(n_components=2, random_state=seed, n_neighbors=10,
                             low_memory=True, n_jobs=-1)
    reducer.fit(X[idx_fit])
    emb = reducer.transform(X[idx_eval])
    km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init="auto", batch_size=10_000)
    return km.fit_predict(emb)

## 5. Correr las 3 semillas (baseline vs. indicador vs. MICE)

⏱️ Este paso corre UMAP+K-Means 9 veces (3 variantes × 3 semillas) — puede tardar 20-40 minutos en Colab gratuito. Guarda resultados parciales en cada semilla por si la sesión se desconecta.

In [ ]:
SEEDS = [500, 501, 502]
K, N_FIT, N_EVAL = 8, 80_000, 50_000
resultados = {"seeds": SEEDS, "ari_indicador": [], "ari_mice": []}
results_path = os.path.join(OUT_DIR, "resultados.json")

for seed in SEEDS:
    t_s = time.time()
    rng_fit = np.random.default_rng(seed=seed)
    idx_fit = rng_fit.choice(n_total, size=N_FIT, replace=False)
    rng_eval = np.random.default_rng(seed=seed + 1000)
    idx_eval = rng_eval.choice(n_total, size=N_EVAL, replace=False)

    print(f"[seed={seed}] pipeline baseline (mediana)...")
    labels_base = correr_pipeline(X_full, seed, idx_fit, idx_eval)

    print(f"[seed={seed}] pipeline variante A (indicadores)...")
    labels_ind = correr_pipeline(X_indicador, seed, idx_fit, idx_eval)
    ari_ind = adjusted_rand_score(labels_base, labels_ind)

    print(f"[seed={seed}] imputación MICE...")
    ord_mice = imputar_mice(seed=seed)
    X_mice = np.hstack([ord_mice, X_full[:, 5:]])
    labels_mice = correr_pipeline(X_mice, seed, idx_fit, idx_eval)
    ari_mice = adjusted_rand_score(labels_base, labels_mice)

    resultados["ari_indicador"].append(float(ari_ind))
    resultados["ari_mice"].append(float(ari_mice))
    json.dump(resultados, open(results_path, "w"), indent=2)
    print(f"[seed={seed}] ARI base-vs-indicador={ari_ind:.4f}  "
          f"base-vs-MICE={ari_mice:.4f}  ({time.time()-t_s:.1f}s)")

## 6. Resumen final

In [ ]:
ari_ind_arr = np.array(resultados["ari_indicador"])
ari_mice_arr = np.array(resultados["ari_mice"])
print(f"% faltante: padre={falta_padre.mean()*100:.1f}%  madre={falta_madre.mean()*100:.1f}%")
print(f"ARI base-vs-indicador = {ari_ind_arr.mean():.4f} ± {ari_ind_arr.std():.4f}")
print(f"ARI base-vs-MICE      = {ari_mice_arr.mean():.4f} ± {ari_mice_arr.std():.4f}")
print("\nValores esperados (manuscrito): indicador≈0.464±0.005, MICE≈0.274±0.033")

## 7. Figura (Supplementary Figure S5)
Reproduce la Figura S5. La primera barra ("solo variar semilla") es la referencia de estabilidad del pipeline completo (Secci\u00f3n 5.6 / notebook `estabilidad_pipeline_colab.ipynb`, ARI = 0.506 \u00b1 0.085); no se recalcula aqu\u00ed, se usa como constante de comparaci\u00f3n.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ari_ind_arr = np.array(resultados["ari_indicador"])
ari_mice_arr = np.array(resultados["ari_mice"])

# Referencia tomada del notebook de estabilidad (mismo tratamiento de datos
# faltantes, solo variando semilla) -- no se recalcula en este notebook.
ARI_REF_SOLO_SEMILLA = 0.506
ARI_REF_SD = 0.085

fig, ax = plt.subplots(figsize=(7.5, 5.8), dpi=150)
labels = ['Solo variar semilla\n(mismo tratamiento\nde datos faltantes)',
          'Mediana vs.\nindicador de\nfaltante',
          'Mediana vs.\nimputaci\u00f3n\nm\u00faltiple (MICE)']
means = [ARI_REF_SOLO_SEMILLA, ari_ind_arr.mean(), ari_mice_arr.mean()]
errs = [ARI_REF_SD, ari_ind_arr.std(), ari_mice_arr.std()]
colors = ['#A0A0A0', '#3B6FA0', '#B33F3F']

bars = ax.bar(labels, means, yerr=errs, capsize=6, color=colors, alpha=0.85)
ax.axhline(ARI_REF_SOLO_SEMILLA, color='gray', linestyle='--', linewidth=1)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, m + 0.03, f'{m:.3f}', ha='center', fontsize=10)
ax.set_ylabel('ARI (acuerdo entre particiones)')
ax.set_title('Efecto del tratamiento de datos faltantes\nen educaci\u00f3n de los padres sobre las asignaciones de cl\u00faster')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_datos_faltantes.png'), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("\u2705 Figura guardada en Drive (Supplementary Figure S5)")
